<a href="https://colab.research.google.com/github/missstechie/Online-Internship-I-HUB-Data-IIITH-Vision-Tasks-using-Generative-AI/blob/main/Documentation_of_LoRA_Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indian Legal Question Answering using LoRA Fine-Tuning

## Project Summary

| Property          | Details                                             |
| :---------------- | :-------------------------------------------------- |
| **Project Title** | Indian Legal Question Answering using LoRA Fine-Tuning |
| **Objective**     | Explore parameter-efficient fine-tuning for Indian legal Q&A |
| **Domain**        | Indian Legal Question Answering                     |
| **Base Model**    | Qwen2.5-7B                                          |
| **Fine-Tuning**   | LoRA with 4-bit quantization (QLoRA-style)          |
| **Dataset**       | Indian Legal Data v2 (5,000 examples)               |
| **Platform**      | Google Colab, NVIDIA Tesla T4                       |
| **Outcome**       | Demonstrated domain-specific fine-tuning pipeline   |


## Project Workflow

```mermaid
graph LR
    A[Indian Legal Dataset] --> B[5,000 Training Examples]
    B --> C[Data Formatting]
    C --> D[Qwen2.5-7B]
    D --> E[4-bit Loading]
    E --> F[LoRA Adapters]
    F --> G[60-Step Fine-Tuning]
    G --> H[Indian Legal Q&A Model]
    H --> I[Evaluation]
    I --> J[Before vs After Comparison]
```

---

## 1. Title and Project Overview

### Indian Legal Question Answering using LoRA Fine-Tuning

**Objective:**
This project explores the domain-specific fine-tuning of the **Qwen2.5-7B** language model using **Low-Rank Adaptation (LoRA)** on an **Indian legal instruction-response dataset**. The goal is to investigate how parameter-efficient fine-tuning can adapt a general-purpose language model to the specific needs of the Indian legal question-answering domain.

**Domain:** Indian Legal Question Answering

**Base Model:** Qwen2.5-7B

**Fine-Tuning Method:** LoRA with 4-bit quantization (QLoRA-style setup)

**Dataset:** Indian Legal Data v2

**Platform:** Google Colab with NVIDIA Tesla T4 GPU


---

## 2. Introduction

Large Language Models (LLMs) are powerful artificial intelligence models trained on vast amounts of text data, enabling them to understand, generate, and respond to human language across a wide range of topics. While general-purpose LLMs like Qwen2.5-7B possess impressive capabilities, their broad training can sometimes lead to responses that lack the specific nuances, terminology, or structured format required for highly specialized domains. Consequently, there's often a need for domain-specific adaptation to tailor these models for particular applications.

Legal question answering is one such critical domain where precision, accuracy, and adherence to specific legal frameworks are paramount. Adapting LLMs for this field can significantly enhance access to legal information and assist legal professionals. This project utilizes **Low-Rank Adaptation (LoRA)**, a parameter-efficient fine-tuning technique, to adapt the Qwen2.5-7B model specifically for Indian legal question answering. LoRA allows for efficient customization of large models without the need to retrain all their parameters, making it a practical approach for resource-constrained environments.

This report documents the process of adapting Qwen2.5-7B to the Indian legal question-answering domain and presents a qualitative comparison between the original base model and the LoRA fine-tuned model.

---

## 3. Problem Statement

General-purpose Large Language Models (LLMs) are trained on diverse and broad datasets, making them versatile but potentially less adept at producing highly specific, nuanced, and structurally appropriate responses for specialized domains. In the context of **Indian Legal Question Answering**, this can manifest as responses that, while generally informative, may not always align with the precise legal terminology, frameworks, or expected output formats prevalent in Indian law.

Full fine-tuning of a large 7-billion-parameter model like Qwen2.5-7B is computationally expensive, requiring significant GPU resources and extended training times. This often makes comprehensive domain adaptation impractical for individual researchers or projects with limited computational budgets. **Low-Rank Adaptation (LoRA)** emerges as a solution by providing a parameter-efficient alternative, allowing for adaptation with a fraction of the computational cost.

This project investigates whether LoRA can effectively adapt the Qwen2.5-7B base model to generate more relevant, structured, and domain-specific answers for Indian legal questions, thereby demonstrating the practicality of fine-tuning for specialized applications on platforms like Google Colab with consumer-grade GPUs.

---

## 4. Dataset

The project utilized **Indian Legal Data v2**, a specialized dataset focused on Indian legal instruction-response pairs. This dataset is crucial for teaching the model to understand and generate content within the Indian legal context.

### Dataset Properties

| Property          | Details                 |
| :---------------- | :---------------------- |
| **Dataset Name**  | Indian Legal Data v2    |
| **Total Examples**| 171,640                 |
| **Fields**        | `instruction`, `response` |
| **Training Subset**| 5,000                   |
| **Domain**        | Indian Legal Question Answering |


The dataset consists of two primary fields:
*   `instruction`: Represents a legal question or a specific instruction related to Indian law.
*   `response`: Contains the corresponding legal answer or response to the instruction.

Before training, the entire dataset was shuffled using a seed of `3407` to ensure randomness and prevent any ordering bias. For this initial experiment, a subset of **5,000 examples** was selected from the total 171,640 examples. These 5,000 examples were then converted into a single text field, concatenating the instruction and response for model training. An End-Of-Sentence (EOS) token was added at the end of each example to signal the completion of a response to the model.

**Reason for 5,000 Examples:** The purpose of using only 5,000 examples for this initial experiment was to conduct a manageable first training run on a Google Colab instance equipped with a NVIDIA Tesla T4 GPU. This allowed for an efficient exploration of the fine-tuning pipeline without immediately committing to the extensive computational resources and time that would be required to train on the entire 171,640 examples.

---

## 5. Dataset Preparation

To effectively fine-tune the Qwen2.5-7B model, the selected 5,000 examples from the Indian Legal Data v2 dataset were prepared into a specific instruction-response format. This structured format helps the model learn the desired input-output pattern for legal question answering.

The training data was formatted as follows:

```
Below is an instruction related to Indian law.
Provide a clear and informative legal response.

Instruction:

[legal question]

Response:

[legal answer]
```

In this format:
*   The original `instruction` field from the dataset was mapped to `[legal question]`.
*   The original `response` field from the dataset was mapped to `[legal answer]`.

Each formatted example, including the instruction and its corresponding response, was then concatenated into a single text string. An End-Of-Sentence (EOS) token was appended to the end of each complete example to clearly delineate individual training instances for the model. This resulting text was stored in a dedicated text column, which was then fed to the trainer for supervised fine-tuning.

---

## 6. Model and Training Configuration

This section details the base model selected and the specific configuration parameters used for the LoRA fine-tuning process.

### Training Configuration Summary

| Parameter                 | Value               |
| :------------------------ | :------------------ |
| **Base Model**            | Qwen2.5-7B          |
| **Quantization**          | 4-bit               |
| **Maximum Sequence Length**| 2048                |
| **LoRA Rank (r)**         | 16                  |
| **LoRA Alpha**            | 16                  |
| **LoRA Dropout**          | 0                   |
| **Training Examples**     | 5,000               |
| **Training Steps**        | 60                  |
| **GPU**                   | NVIDIA Tesla T4     |
| **Available GPU Memory**  | ~14.56 GB           |
| **Peak Reserved GPU Memory** | 10.344 GB           |
| **Training Loss**         | 1.0021              |
| **Training Runtime**      | ~1067 seconds       |


**What LoRA Does:**
Low-Rank Adaptation (LoRA) is a parameter-efficient fine-tuning technique that significantly reduces the number of trainable parameters when adapting a large pre-trained model to a new task. Instead of updating all the weights of the massive Qwen2.5-7B model, LoRA introduces small, trainable adapter matrices (low-rank matrices) into specific layers of the pre-trained model. During fine-tuning, only these small adapter matrices are trained, while the original, large pre-trained model weights remain frozen. This approach drastically reduces computational requirements and storage for the fine-tuned model, making it feasible to train on more modest hardware.

**LoRA Target Modules:**
For this project, LoRA adapters were applied to the following key modules within the Qwen2.5-7B model's architecture:
*   `q_proj` (query projection)
*   `k_proj` (key projection)
*   `v_proj` (value projection)
*   `o_proj` (output projection)
*   `gate_proj` (gate projection in MLP)
*   `up_proj` (up-projection in MLP)
*   `down_proj` (down-projection in MLP)

These modules are critical components of the transformer's attention mechanism and feed-forward networks, and targeting them allows for effective adaptation of the model's feature representation capabilities.

**4-bit Quantization (QLoRA-style Setup):**
**4-bit quantization** was enabled for loading the base model. This technique reduces the precision of the model's weights from higher precision (e.g., 16-bit or 32-bit floating-point) to 4-bit integers. By doing so, it significantly cuts down the GPU memory requirements needed to load and process the large base model. This memory efficiency was crucial for performing the fine-tuning on a Google Colab environment with an NVIDIA Tesla T4 GPU, which has a limited amount of available video memory (~14.56 GB).

---

## 7. Training Results

The LoRA fine-tuning experiment was conducted on the Qwen2.5-7B model using 5,000 examples from the Indian Legal Data v2 dataset. The training run completed successfully, yielding the following key results:

*   **Training steps:** 60
*   **Training loss:** 1.0021188348531722
*   **Training runtime:** 1067.0726 seconds (approximately 17.8 minutes)
*   **Peak reserved GPU memory:** 10.344 GB
*   **GPU:** NVIDIA Tesla T4

Training completed without any errors, indicating a successful execution of the fine-tuning pipeline. It's important to note that the reported training loss of **1.0021** is an indicator of how well the model fit the training objective during this short experiment. It does **not** directly translate to the model's accuracy or overall performance on unseen legal questions. Given the very limited number of training steps (60), this loss value primarily reflects the initial adaptation phase.

Upon completion, the LoRA adapter weights were successfully saved to the following directory:

```
lora_model/
```
This saved adapter can be loaded and merged with the base Qwen2.5-7B model for inference, allowing for efficient deployment of the fine-tuned capabilities.

---

## 8. Model Evaluation

To qualitatively assess the impact of the LoRA fine-tuning, the following five Indian legal questions were used to probe both the original Qwen2.5-7B base model and the LoRA fine-tuned model:

1.  What is Article 32 of the Indian Constitution?
2.  What is judicial review in India?
3.  What is the purpose of a writ petition?
4.  What is the difference between a civil case and a criminal case?
5.  What is Article 21 of the Indian Constitution?

This approach provides a simple, qualitative **before-vs-after comparison** to observe any immediate changes in the model's responses after the fine-tuning process. The same set of questions ensures a consistent basis for comparison between the two model versions.

---

## 9. Before vs After Comparison

Qualitative observations from testing the base Qwen2.5-7B model and the LoRA fine-tuned model with the five test questions are summarized below:

| Test Question                      | Base Model (Qwen2.5-7B)                  | LoRA Model (Fine-tuned)                      |
| :--------------------------------- | :--------------------------------------- | :------------------------------------------- |
| **What is Article 32?**            | Detailed and structured answer           | Relevant answer but sometimes repetitive     |
| **What is judicial review?**       | Detailed legal explanation               | Relevant legal explanation with some repetition |
| **Purpose of a writ petition?**    | Legal explanation, somewhat narrow       | Relevant legal response                      |
| **Civil vs criminal case?**        | Structured and detailed                  | Structured legal response                    |
| **What is Article 21?**            | Clear explanation                        | Relevant answer but sometimes repetitive     |


### Observations and Discussion:

The original Qwen2.5-7B base model already possessed substantial general knowledge of Indian legal and constitutional concepts, producing reasonably strong answers to the test questions. The short 60-step LoRA fine-tuning experiment, while successful in adapting the model's weights, **did not consistently demonstrate a significant improvement in answer quality across all five questions**. In some instances, the LoRA model's responses, particularly for Article 32 and Article 21, exhibited repetitive text, suggesting that the limited training might have led to some overfitting or a lack of generalization for certain concepts.

Furthermore, some answers from the LoRA model were truncated, primarily because the `max_new_tokens` parameter for generation was fixed at 256, which might not have been sufficient for comprehensive legal explanations.

**Therefore, it is crucial not to claim that the LoRA model was objectively better than the base model based on these qualitative observations.** Instead, this experiment successfully demonstrated the practical aspects of the domain-specific LoRA fine-tuning pipeline, including:
*   Dataset preparation for domain adaptation.
*   Application of domain-specific formatting.
*   Successful LoRA adapter creation and integration.
*   Execution of the training process.
*   Saving of the fine-tuned model adapter.
*   Performing a basic before-vs-after qualitative comparison.

This comparison is an **experimental observation**, serving to validate the fine-tuning process rather than a formal benchmark of improved legal answer quality.

---

## 10. Features of the Fine-Tuned Model

The LoRA fine-tuned Qwen2.5-7B model, while still an experimental prototype, exhibits several key features derived from the adaptation process:

1.  **Indian Legal Domain Specialization:** The model has been fine-tuned using specific Indian legal instruction-response examples, aiming to align its knowledge and generation style with the Indian legal context.
2.  **Legal Question Answering Capability:** It is designed to generate relevant and informative responses to questions pertaining to Indian law, constitutional provisions, and legal concepts.
3.  **Instruction-Response Format Adherence:** The model was trained to follow a structured legal question and answer format, improving the readability and clarity of its outputs.
4.  **LoRA-Based Adaptation:** Utilizes lightweight trainable adapters, meaning only a small fraction of the model's parameters were updated, making the fine-tuning process memory and computationally efficient.
5.  **Memory-Efficient Fine-Tuning:** By leveraging 4-bit model loading, the model was successfully trained on a Google Colab environment with a NVIDIA Tesla T4 GPU, demonstrating feasibility on limited hardware resources.
6.  **Potential for Structured Legal Responses:** The fine-tuning aims to encourage the model to produce responses with clear headings, numbered explanations, and appropriate legal terminology, mimicking professional legal documentation.
7.  **Reusable LoRA Adapter:** The fine-tuned LoRA adapter weights are saved separately (to `lora_model/`), allowing for flexible integration with the original Qwen2.5-7B base model without modifying its core weights.

---

## 11. Limitations

It is important to acknowledge the limitations of this experimental project, which are crucial for interpreting the results and planning future work:

1.  **Short Training Run:** Only 60 training steps were executed. This is a very limited number for a large language model and is generally insufficient to achieve comprehensive domain adaptation or significant performance improvements.
2.  **Limited Training Subset:** Only 5,000 examples were used from the total 171,640 available in the Indian Legal Data v2 dataset. A larger and more diverse training set would likely lead to better generalization and stronger performance.
3.  **Repetitive Responses:** Some outputs from the fine-tuned model, particularly for specific constitutional articles, exhibited repetitive text. This is a common issue with limited fine-tuning and can indicate insufficient exposure to diverse data or inadequate training duration.
4.  **Truncated Responses:** Certain responses were truncated (cut off prematurely) because the `max_new_tokens` parameter was set to 256 during inference. This limits the verbosity and completeness of the generated legal answers.
5.  **No Formal Accuracy Benchmark:** The evaluation was purely qualitative, relying on five manually selected test questions. There was no formal accuracy metric, benchmark score, or quantitative evaluation conducted, making it impossible to definitively claim improved performance.
6.  **Base Model Already Has Legal Knowledge:** The Qwen2.5-7B base model already demonstrated a reasonable understanding of general Indian legal concepts. This makes it challenging for a short fine-tuning run to show dramatic improvements, as the model starts from an already capable state.
7.  **Legal Reliability:**
    
    <div style="background-color:#ffe0b2; padding: 15px; border-radius: 5px; border: 1px solid #ff9800;">
      ⚠️ **IMPORTANT: The model developed in this project is an educational and research prototype. It should NOT be treated as, relied upon for, or substituted for professional legal advice. Legal accuracy is not guaranteed, and users should always consult qualified legal professionals for specific legal guidance.**
    </div>
    


---

## 12. Conclusion

This project successfully demonstrated a practical pipeline for adapting the **Qwen2.5-7B** large language model for Indian legal question answering using **Low-Rank Adaptation (LoRA)**. The fine-tuning process leveraged the **Indian Legal Data v2** dataset, utilizing a subset of 5,000 examples selected from the total 171,640 available. Despite the brief training duration of only 60 steps, the process completed successfully on a **NVIDIA Tesla T4 GPU** within Google Colab, facilitated by **4-bit quantization** for memory efficiency. The LoRA adapter was successfully saved, enabling future use.

Qualitative evaluation was performed using five specific Indian legal questions, providing a **before-vs-after comparison** against the base model. While this short experimental run was not sufficient to conclusively prove a significant improvement in legal answer quality or eliminate issues like repetitive text, it effectively showcased the methodology for domain-specific fine-tuning.

More extensive training, a larger and more diverse dataset, and a rigorous formal evaluation with quantitative metrics would be necessary to draw stronger conclusions regarding the model's accuracy and practical utility for legal applications.

Overall, this project demonstrates how parameter-efficient fine-tuning using LoRA can be used to explore domain-specific adaptation of a large language model for Indian legal question answering using limited computational resources.

---

## 13. Technology Stack

This project utilized the following technologies and libraries:

| Technology            | Purpose                                        |
| :-------------------- | :--------------------------------------------- |
| **Google Colab**      | Cloud-based development and training environment |
| **NVIDIA Tesla T4**   | GPU acceleration for model training            |
| **Qwen2.5-7B**        | Base large language model                      |
| **Unsloth**           | Efficient model loading and fine-tuning        |
| **LoRA**              | Parameter-efficient fine-tuning technique      |
| **4-bit Quantization**| Memory-efficient model loading                 |
| **Hugging Face Datasets**| Dataset loading and processing             |
| **Hugging Face TRL**  | Library for Transformer Reinforcement Learning (used for supervised fine-tuning) |
| **PyTorch**           | Deep learning framework                        |
